# Polynomial Regression - Practical, Explained

**Goal:** when dots make a curve instead of a straight line, give the model extra columns such as $x^2$, then let Linear Regression learn a curved prediction.

## 1. The tiny idea

A straight-line model uses: $\hat{y}=b_0+b_1x$.

For a U-shaped pattern, degree 2 adds a squared copy of the same input: $\hat{y}=b_0+b_1x+b_2x^2$.

Think of $x^2$ as a new clue for the model. The model is still **linear in the weights** $b_0, b_1, b_2$; its picture becomes curved because $x^2$ bends.

## 2. Plan before code

1. Make a small curved dataset.
2. Keep some dots hidden for testing.
3. Compare a straight-line model (degree 1) with a curve (degree 2).
4. Check the score on the hidden test dots—not only on the dots used for learning.

**Important:** `fit_transform(X_train)` learns the feature setup from training data. `transform(X_test)` only applies that same setup to test data. This keeps the test set a fair exam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

# Seed = same random example every time you run the cell.
rng = np.random.default_rng(42)

# X is one input column.  The true pattern is U-shaped: 0.5*x^2 + 1.5*x + 2.
X = rng.uniform(-3, 3, size=(100, 1))
y = 0.5 * X[:, 0] ** 2 + 1.5 * X[:, 0] + 2 + rng.normal(0, 0.7, size=100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

plt.figure(figsize=(7, 4))
plt.scatter(X_train, y_train, label='Training dots', color='#1d3557')
plt.scatter(X_test, y_test, label='Hidden test dots', color='#f4a261')
plt.xlabel('x (input)')
plt.ylabel('y (answer)')
plt.title('Our data looks like a curve, not a straight line')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print(f'Training dots: {len(X_train)} | Hidden test dots: {len(X_test)}')

## 3. First try: a straight line

This model can only tilt up or down. It cannot make a U-shape, so it will miss the bend in the middle.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)          # Learn from training dots only.
linear_test_predictions = linear_model.predict(X_test)

linear_r2 = r2_score(y_test, linear_test_predictions)
linear_rmse = mean_squared_error(y_test, linear_test_predictions) ** 0.5

print(f'Straight line — test R²: {linear_r2:.3f}, test RMSE: {linear_rmse:.3f}')
print('R² closer to 1 is better. RMSE is typical prediction error; smaller is better.')

## 4. Add the squared clue: $x^2$

`PolynomialFeatures(degree=2)` turns one input column `[x]` into `[x, x²]`.

We use `include_bias=False` because `LinearRegression()` already learns the constant/intercept $b_0$. Adding it twice is unnecessary.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)  # Learn the columns from training data.
X_test_poly = poly.transform(X_test)        # Apply exactly the same columns to test data.

print('Feature names:', poly.get_feature_names_out(['x']))
print('For x = 3, the model sees:', poly.transform([[3]])[0])

polynomial_model = LinearRegression()
polynomial_model.fit(X_train_poly, y_train)
poly_test_predictions = polynomial_model.predict(X_test_poly)

poly_r2 = r2_score(y_test, poly_test_predictions)
poly_rmse = mean_squared_error(y_test, poly_test_predictions) ** 0.5

print(f'Polynomial degree 2 — test R²: {poly_r2:.3f}, test RMSE: {poly_rmse:.3f}')

## 5. Read the picture

- **Red line:** too simple for a curved pattern (underfitting).
- **Green curve:** follows the main U-shape without trying to touch every dot.
- Dots never need to sit exactly on the curve—small random noise is normal.

In [ ]:
# Make x values in order so model predictions draw smooth lines.
X_draw = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='#1d3557', alpha=0.75, label='Training dots')
plt.scatter(X_test, y_test, color='#f4a261', alpha=0.90, label='Hidden test dots')
plt.plot(X_draw, linear_model.predict(X_draw), color='#d62828', linewidth=2.5,
         label=f'Straight line (R²={linear_r2:.2f})')
plt.plot(X_draw, polynomial_model.predict(poly.transform(X_draw)), color='#2a9d8f', linewidth=2.8,
         label=f'Degree-2 curve (R²={poly_r2:.2f})')
plt.xlabel('x (input)')
plt.ylabel('y (answer)')
plt.title('Polynomial Regression captures the bend')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 6. Do not keep increasing the degree

A large degree (for example, 10) can wiggle to chase random bumps in the training dots. It may look clever in training but make poor predictions for new dots. This is **overfitting**.

Use a validation set or cross-validation to compare degree 1, 2, 3, ... Choose the simplest degree that performs well on unseen data.

## Quick revision

- Curved data can need Polynomial Regression.
- Degree 2 creates $x^2$; degree 3 also creates $x^3$.
- Split first. Fit feature changes and the model on training data only.
- Compare test R² (higher is better) and test RMSE (lower is better).
- Pick the smallest degree that generalizes well.